# Bab 4: Interpretasi & Evaluasi Hasil Model Klasifikasi Hand Gesture

Dokumen ini berisi analisis dan interpretasi mendalam mengenai performa model AI yang telah dilatih pada tahap pemodelan (`02_modeling.ipynb`). Evaluasi ini dilakukan untuk membedah hasil metrik akurasi, grafik *Confusion Matrix*, serta laporan klasifikasi detail dari dua arsitektur model, yaitu **Random Forest** (setelah *Hyperparameter Tuning*) dan **SVM Linear** (*Baseline*).

## 4.1 Analisis Akurasi Keseluruhan (Accuracy)

Berdasarkan hasil pengujian menggunakan dataset sebesar 20.000 sampel data latih (*training*) dan 5.000 sampel data uji (*testing*), diperoleh performa akurasi sebagai berikut:

| Model Klasifikasi | Kondisi Model | Akurasi Pengujian (Testing Accuracy) | Waktu Komputasi |
| :--- | :--- | :--- | :--- |
| **Random Forest** | Setelah *Hyperparameter Tuning* | **82.03%** | ~ 5 - 10 Menit |
| **SVM Linear** | *Baseline* (Tanpa Tuning) | **93.55%** | ~ 32 Menit |


### Analisis Angka:
Model **SVM Linear** secara mengejutkan berhasil mencapai akurasi tertinggi sebesar **93.55%**. Hal ini membuktikan bahwa karakteristik data koordinat *hand landmarks* ($X, Y, Z$) yang diekstrak menggunakan MediaPipe memiliki sifat *linearly separable* (fitur antar kelas gestur dapat dipisahkan dengan sangat baik oleh garis pembatas atau *hyperplane* geometris milik SVM).

In [ ]:
from IPython.display import Image, display

# Langsung tembak file gambar menggunakan fungsi Image bawaan
display(Image(filename="../app/assets/images/bukti_akurasi_RF.png"))

In [ ]:
from IPython.display import Image, display

# Langsung tembak file gambar menggunakan fungsi Image bawaan
display(Image(filename="../app/assets/images/bukti_akurasi_SVM.png"))

## 4.2 Analisis Visual Confusion Matrix

*Confusion Matrix* digunakan untuk melihat seberapa baik model dalam memprediksi tiap-tiap kelas gestur secara spesifik dan mendeteksi di mana letak salah tebak (*misclassification*) terbesar terjadi.

Berikut adalah bukti visual hasil running grafik *Confusion Matrix* serta akurasi tertinggi yang berhasil didapatkan oleh model SVM Linear (93.55%) pada laptop Ryzen 5:


### 1. Pola Diagonal Utama
Berdasarkan gambar di atas, angka tertinggi terakumulasi pada **diagonal utama** (dari pojok kiri atas ke kanan bawah). Secara visual, hal ini ditunjukkan dengan warna biru pekat di sepanjang jalur diagonal. Ini membuktikan bahwa mayoritas sampel data berhasil diprediksi secara tepat (*True Positive*) oleh AI sesuai dengan label aslinya (*Ground Truth*).

### 2. Analisis Kasus Salah Tebak
Meskipun akurasi keseluruhan SVM lebih tinggi, terdapat pola sebaran eror yang berbeda pada grafisnya:
* **Pada Random Forest:** Pola salah tebak tersebar tipis dan merata di luar diagonal utama (warna biru samar-samar). Tidak ada satu kelas pun yang mendominasi kesalahan.
* **Pada SVM:** Pola salah tebak sangat minim di hampir seluruh kelas (membuat akurasinya tinggi), namun terdapat kotak dengan warna biru yang sedikit lebih kontras pada irisan kelas tertentu. Hal ini menandakan SVM memiliki kelemahan kecil pada gestur yang posisi koordinat jarinya beririsan sangat rapat secara spasial.

In [ ]:
from IPython.display import Image, display

# Langsung tembak file gambar menggunakan fungsi Image bawaan
display(Image(filename="../app/assets/images/bukti_SVM.png"))

## 4.3 Analisis Classification Report Detail

Evaluasi diperdalam dengan melihat nilai *Precision*, *Recall*, dan *F1-Score* untuk memastikan model tidak mengalami ketimpangan prediksi (*bias*):

* **Precision (Presisi):** Nilai presisi yang tinggi (terutama pada SVM yang mencapai rata-rata di atas 0.93) menunjukkan tingkat kepastian prediksi yang sangat andal. Ketika AI menebak sebuah gestur adalah "A", maka probabilitas tebakan itu benar sangat tinggi.
* **Recall (Sensitivitas):** Nilai *recall* yang seimbang membuktikan model mampu menyaring dan menemukan kembali hampir seluruh sampel dari tiap-tiap kelas gestur yang ada di dalam dataset tanpa banyak data yang terlewat.
* **F1-Score:** Karena nilai *Precision* dan *Recall* berjalan seimbang dan tinggi, nilai *F1-Score* yang dihasilkan pun optimal. Ini mengindikasikan bahwa model tidak hanya pintar menebak satu atau dua gestur dominan, melainkan merata di seluruh variasi gerakan tangan.

In [ ]:
from IPython.display import Image, display

# Langsung tembak file gambar menggunakan fungsi Image bawaan
display(Image(filename="../app/assets/images/Laporan-klasifikasi.png"))

## 4.4 Analisis Kritis: Fenomena Visual vs Angka Akurasi

Terdapat temuan menarik di mana secara visual grafis *Confusion Matrix* milik Random Forest terlihat lebih "rapi dan seimbang", namun secara angka akurasi murni SVM jauh lebih unggul. 

Hal ini terjadi karena karakteristik algoritma:
1. **Random Forest** bekerja berbasis kumpulan pohon keputusan (*ensemble*) yang membagi keputusan secara bertahap, sehingga hasil erornya terdistribusi merata ke seluruh cabang kelas.
2. **SVM Linear** bekerja dengan menarik garis batas kaku (*hyperplane*). Selama mayoritas data berada di sisi garis yang benar, akurasinya akan melonjak drastis (93.55%). Namun, jika ada beberapa koordinat jari dari kelas lain yang tidak sengaja melewati garis tersebut, mereka akan langsung salah diklasifikasikan secara massal pada satu titik kotak tertentu di luar diagonal.

## 4.5 Justifikasi Akhir Pemilihan Model untuk Deployment (Streamlit)

Berdasarkan analisis *trade-off* antara akurasi, performa visual, dan waktu komputasi, diambil keputusan sebagai berikut:

1. **Model yang Dipilih:** **SVM Linear (Akurasi 93.55%)**.
2. **Justifikasi:** * Aplikasi Streamlit yang akan dibangun membutuhkan tingkat akurasi prediksi tertinggi agar pengguna dapat berinteraksi dengan gestur tangan secara presisi tanpa hambatan salah deteksi.
   * Meskipun proses *training* SVM memakan waktu sangat lama (32 menit) pada dataset 20.000 sampel, proses pelatihan ini hanya dilakukan sekali di awal secara lokal (*offline training*). 
   * Begitu model diekspor menjadi file `.pkl`, proses prediksi (*inference*) di dalam aplikasi Streamlit akan berjalan sangat ringan dan instan (dalam hitungan milidetik), sehingga waktu *training* yang lama tidak akan memengaruhi performa aplikasi akhir.